# HyDE — Hypothetical Document Embeddings

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/HyDe_Hypothetical_Document_Embedding.ipynb)

## A Deep Dive into Bridging the Query-Document Gap

**Paper**: *Precise Zero-Shot Dense Retrieval without Relevance Labels* — Gao et al., 2022

HyDE is an elegantly simple idea that dramatically improves dense retrieval: instead of embedding a short query and hoping it lands near the right documents, we ask an LLM to **imagine** an answer first, then use *that* imagined answer as the search query. This notebook builds HyDE from scratch, explains *why* it works geometrically, and compares it head-to-head with standard retrieval.

<div style="text-align: center;">

<img src="../images/HyDe.svg" alt="HyDe" style="width:40%; height:auto;">
</div>

## 1 · The Query-Document Gap

### Why Short Queries Fail in Dense Retrieval

When you ask *"What causes the Northern Lights?"*, your query is **7 words**. The document that answers it might be a **300-word paragraph** explaining solar wind, the magnetosphere, atmospheric gases, and photon emission.

**The core problem:** Embedding models compress text into a fixed-size vector (e.g., 768 dimensions). A 7-word question and a 300-word explanation occupy **very different regions** of embedding space, even when they are semantically related.

| Property | User Query | Relevant Document |
|---|---|---|
| Length | 5–15 words | 50–500 words |
| Style | Interrogative | Declarative / expository |
| Vocabulary | Colloquial, vague | Technical, specific |
| Information density | Low | High |

This mismatch is called the **query-document gap**. Standard dense retrieval tries to bridge it during training by learning a shared embedding space, but the gap persists — especially for out-of-domain queries the model has never seen.

### Cosine Similarity: The Geometric View

Cosine similarity measures the **angle** between two vectors. Two texts that use similar words in similar patterns will have high cosine similarity. But:

- *"What causes auroras?"* → embeds near other short questions
- *"Solar wind particles interact with Earth's magnetosphere..."* → embeds near other scientific explanations

They live in **different neighborhoods** of the embedding space, even though one answers the other.

In [ ]:
# Let's quantify the query-document gap with a concrete example
# We'll use a simple embedding model to show how far apart queries and docs really are

from sentence_transformers import SentenceTransformer
import numpy as np

# A small model just for this quick demo (we'll load the full model later)
demo_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

query = "What causes the Northern Lights?"
document = (
    "The Northern Lights, or aurora borealis, are caused by charged particles from the Sun "
    "interacting with Earth's magnetosphere. Solar wind carries electrons and protons that "
    "collide with atmospheric gases — primarily oxygen and nitrogen — at altitudes of 100 to "
    "300 km. These collisions excite gas molecules, which emit photons of visible light as "
    "they return to their ground state. The colors depend on the gas species and altitude: "
    "green from oxygen at lower altitudes, red from oxygen at higher altitudes, and blue-purple "
    "from nitrogen."
)
hypothetical_answer = (
    "The Northern Lights are caused by solar wind particles — electrons and protons ejected "
    "from the Sun — colliding with gas molecules in Earth's upper atmosphere. These collisions "
    "excite atmospheric gases like oxygen and nitrogen, which then emit colorful light as they "
    "return to their ground state."
)

embeddings = demo_model.encode([query, document, hypothetical_answer])
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

sim_query_doc = cosine_sim(embeddings[0], embeddings[1])
sim_hypo_doc  = cosine_sim(embeddings[2], embeddings[1])
sim_query_hypo = cosine_sim(embeddings[0], embeddings[2])

print("="*60)
print("QUERY-DOCUMENT GAP — Cosine Similarities")
print("="*60)
print(f"  Query  ↔ Document:       {sim_query_doc:.4f}")
print(f"  HypoDoc ↔ Document:     {sim_hypo_doc:.4f}")
print(f"  Query  ↔ HypoDoc:       {sim_query_hypo:.4f}")
print(f"\n  Gap closed by HyDE:     +{sim_hypo_doc - sim_query_doc:.4f}")
print("="*60)
print("\nThe hypothetical document is MUCH closer to the real document")
print("than the original query is. That's the entire idea behind HyDE.")

del demo_model  # free memory for the main model

## 2 · What is HyDE?

### Hypothetical Document Embeddings (Gao et al., 2022)

HyDE stands for **Hy**pothetical **D**ocument **E**mbeddings. The key insight is deceptively simple:

> **Don't embed the query. Embed a hypothetical answer to the query.**

Here's the algorithm:

1. **User asks a question** → *"What causes the Northern Lights?"*
2. **LLM generates a hypothetical document** that would answer the question — it doesn't need to be factually correct!
3. **Embed the hypothetical document** instead of the original query
4. **Search the vector store** using the hypothetical document's embedding
5. **Retrieve real documents** that are similar to the hypothetical one
6. **Generate the final answer** from the *real* retrieved documents (not the hypothetical one)

### Why "Hypothetical"?

The generated document is explicitly *hypothetical* — it may contain errors, hallucinations, or incomplete information. That's perfectly fine! Its purpose is not to provide the answer, but to **act as a better search query** by being closer in embedding space to real documents.

Think of it this way: the LLM acts as a **query expansion engine** that transforms a short question into a long, document-like text. The embedding model then does what it does best — finding similar documents.

<div style="text-align: center;">

<img src="../images/hyde-advantages.svg" alt="HyDE Advantages" style="width:100%; height:auto;">
</div>

## 3 · Why HyDE Works — The Geometric Argument

### Documents Live Near Other Documents

Embedding models are trained on **document-document** similarity far more than **query-document** similarity. This means:

- Documents about the same topic cluster together
- Questions about that topic are *nearby* but not *inside* the cluster

A hypothetical document, even an imperfect one, lands **inside the document cluster** because it:
- Uses **declarative, expository language** (like real documents)
- Has **similar length** to real documents
- Contains **domain vocabulary** that the embedding model associates with the topic
- Follows the **structural patterns** the embedding model learned during training

### The LLM as a "Space Translator"

The LLM doesn't need to produce a correct answer. It needs to produce text that **sounds like a document on the right topic**. Even a hallucinated explanation of the Northern Lights will use words like "solar wind", "magnetosphere", "atmosphere", "photons" — words that pull the embedding toward the correct region of vector space.

### Formal Intuition

Let `d*` be the ideal document, `q` be the query, and `h` be the hypothetical document:

- `cos(embed(q), embed(d*))` is limited by the query-document gap
- `cos(embed(h), embed(d*))` is much higher because `h` and `d*` are both documents on the same topic
- Even if `h` contains factual errors, its **embedding** is still close to `d*`'s embedding

## 4 · The HyDE Algorithm — Step by Step

```
┌─────────────────┐
│   User Query     │  "What causes the Northern Lights?"
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   LLM Generation │  Generate a hypothetical answer (~200 words)
│   (Qwen3-8B)    │  "The Northern Lights are caused by solar wind..."
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   Embed HypoDoc  │  BAAI/bge-base-en-v1.5 → 768-dim vector
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   FAISS Search    │  Find top-k nearest real documents
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   Retrieved Docs  │  Real, factual documents from the knowledge base
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   Final Answer    │  LLM generates answer from REAL docs only
└─────────────────┘
```

**Key point:** The hypothetical document is used ONLY for retrieval. The final answer is generated from real, verified documents. This means HyDE inherits the factuality of the knowledge base, not the LLM's hallucinations.

## 5 · Setup — Loading Models

We load two models:
1. **Qwen3-8B** (4-bit NF4 quantized) — for generating hypothetical documents and final answers
2. **BAAI/bge-base-en-v1.5** — for computing document embeddings

Both are cached to Google Drive so they persist across Colab sessions.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch
!pip install -q sentence-transformers faiss-cpu numpy

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=2048, temperature=0.7, do_sample=True, thinking=True):
    """Generate a response using Qwen3, optionally with native thinking mode."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=thinking,
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    gen_kwargs = dict(
        max_new_tokens=max_new_tokens, do_sample=do_sample, top_k=20,
        temperature=temperature if do_sample else None,
        top_p=0.95 if thinking else 0.8,
    )
    output_ids = model.generate(**inputs, **gen_kwargs)
    token_ids = output_ids[0][inputs.input_ids.shape[1]:].tolist()
    if thinking:
        try:
            idx = len(token_ids) - token_ids[::-1].index(151668)
        except ValueError:
            idx = 0
        thought = tokenizer.decode(token_ids[:idx], skip_special_tokens=True).strip()
        answer  = tokenizer.decode(token_ids[idx:], skip_special_tokens=True).strip()
        return thought, answer
    return tokenizer.decode(token_ids, skip_special_tokens=True).strip()

print(f"✓ LLM loaded: {MODEL_NAME} (4-bit NF4)")

In [ ]:
# Load the embedding model
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
embed_model = SentenceTransformer(EMBED_MODEL_NAME, cache_folder=CACHE_DIR)

# Verify it works
test_emb = embed_model.encode(["Hello world"])
print(f"✓ Embedding model loaded: {EMBED_MODEL_NAME}")
print(f"  Embedding dimension: {test_emb.shape[1]}")
print(f"  dtype: {test_emb.dtype}")

## 6 · Building a Synthetic Knowledge Base

We create a knowledge base about **space science and astronomy** — 20 detailed passages covering diverse topics. This lets us test HyDE on questions where the answer exists in the knowledge base but might be hard to find with a short query.

In [ ]:
# Synthetic knowledge base — diverse space science topics
knowledge_base = [
    {
        "id": "solar_wind_01",
        "text": (
            "Solar wind is a continuous stream of charged particles — primarily protons and "
            "electrons — ejected from the Sun's upper atmosphere (the corona) at speeds ranging "
            "from 300 to 800 km/s. The solar wind carries the Sun's magnetic field into "
            "interplanetary space, forming the heliosphere — a vast bubble that extends well "
            "beyond the orbit of Pluto. When solar wind encounters a planet's magnetosphere, "
            "the interaction can produce spectacular phenomena such as auroras. The intensity "
            "of solar wind varies with the 11-year solar cycle, peaking during solar maximum."
        ),
    },
    {
        "id": "aurora_01",
        "text": (
            "Auroras — the Northern Lights (aurora borealis) and Southern Lights (aurora "
            "australis) — occur when charged particles from the solar wind are channeled by "
            "Earth's magnetic field toward the polar regions. These particles collide with "
            "atmospheric gases at altitudes between 100 and 300 km. Collisions with oxygen "
            "produce green light (557.7 nm) at lower altitudes and red light (630.0 nm) at "
            "higher altitudes. Nitrogen collisions produce blue and purple hues. The shape and "
            "intensity of auroras depend on geomagnetic activity, which is driven by solar storms."
        ),
    },
    {
        "id": "magnetosphere_01",
        "text": (
            "Earth's magnetosphere is a region of space dominated by the planet's magnetic "
            "field, extending from the surface to approximately 65,000 km on the sunward side "
            "and forming a long magnetotail on the night side. The magnetosphere deflects most "
            "of the solar wind, protecting the atmosphere from erosion. The Van Allen radiation "
            "belts — two donut-shaped zones of trapped high-energy particles — sit within the "
            "magnetosphere at distances of 1,000 to 60,000 km from Earth's surface. Geomagnetic "
            "storms occur when coronal mass ejections compress and disturb the magnetosphere."
        ),
    },
    {
        "id": "mars_atmosphere_01",
        "text": (
            "Mars has a thin atmosphere composed primarily of carbon dioxide (95.3%), with "
            "minor amounts of nitrogen (2.7%) and argon (1.6%). The atmospheric pressure at "
            "the surface is about 600 pascals — less than 1% of Earth's sea-level pressure. "
            "Mars lost most of its atmosphere over billions of years because it lacks a global "
            "magnetic field to shield against solar wind stripping. The thin atmosphere means "
            "Mars cannot sustain liquid water on its surface under current conditions, though "
            "evidence of ancient river valleys and lake beds suggests a thicker atmosphere in "
            "the past."
        ),
    },
    {
        "id": "exoplanet_detection_01",
        "text": (
            "Exoplanets — planets orbiting stars other than the Sun — are detected using "
            "several methods. The transit method measures the tiny dimming of starlight as a "
            "planet crosses in front of its host star; NASA's Kepler and TESS missions have "
            "discovered thousands of exoplanets this way. The radial velocity method detects "
            "the star's wobble caused by gravitational pull from an orbiting planet. Direct "
            "imaging captures light from the planet itself but works only for large, bright "
            "planets far from their star. As of 2024, over 5,600 exoplanets have been confirmed."
        ),
    },
    {
        "id": "black_holes_01",
        "text": (
            "Black holes are regions of spacetime where gravity is so extreme that nothing — "
            "not even light — can escape once it crosses the event horizon. Stellar black holes "
            "form when massive stars (above ~25 solar masses) exhaust their nuclear fuel and "
            "undergo gravitational collapse. Supermassive black holes, containing millions to "
            "billions of solar masses, reside at the centers of most galaxies. The first direct "
            "image of a black hole's shadow was captured in 2019 by the Event Horizon Telescope "
            "collaboration, showing the supermassive black hole in galaxy M87."
        ),
    },
    {
        "id": "neutron_stars_01",
        "text": (
            "Neutron stars are the collapsed remnants of massive stars (8–25 solar masses) that "
            "have undergone supernova explosions. They pack roughly 1.4 solar masses into a "
            "sphere only 20 km in diameter, producing densities of about 4×10^17 kg/m³ — "
            "comparable to atomic nuclei. Neutron stars spin rapidly (some at hundreds of "
            "revolutions per second) and possess extremely strong magnetic fields. Pulsars are "
            "neutron stars that emit beams of electromagnetic radiation from their magnetic "
            "poles; as the star rotates, these beams sweep across Earth like a lighthouse."
        ),
    },
    {
        "id": "dark_matter_01",
        "text": (
            "Dark matter is a hypothetical form of matter that does not interact with "
            "electromagnetic radiation — it neither emits, absorbs, nor reflects light. "
            "Its existence is inferred from gravitational effects: galaxy rotation curves show "
            "that stars at the edges of galaxies orbit faster than expected from visible mass "
            "alone, implying a large halo of unseen matter. Dark matter constitutes roughly 27% "
            "of the total mass-energy of the universe, compared to only 5% for ordinary "
            "(baryonic) matter. Leading candidates include WIMPs (weakly interacting massive "
            "particles) and axions, but none have been directly detected as of 2024."
        ),
    },
    {
        "id": "dark_energy_01",
        "text": (
            "Dark energy is the mysterious force driving the accelerating expansion of the "
            "universe. Discovered in 1998 through observations of Type Ia supernovae by two "
            "independent teams, dark energy accounts for approximately 68% of the total "
            "mass-energy content of the universe. The simplest explanation is the cosmological "
            "constant (Λ), representing a uniform energy density filling all of space. "
            "Alternative models include quintessence — a dynamic scalar field that evolves over "
            "time. Understanding dark energy is one of the foremost challenges in modern "
            "cosmology and fundamental physics."
        ),
    },
    {
        "id": "cosmic_microwave_background_01",
        "text": (
            "The Cosmic Microwave Background (CMB) is the thermal radiation left over from "
            "the early universe, emitted approximately 380,000 years after the Big Bang when "
            "the universe cooled enough for electrons and protons to combine into neutral "
            "hydrogen (recombination era). The CMB has a nearly perfect blackbody spectrum at "
            "a temperature of 2.725 K and is remarkably uniform across the sky, with tiny "
            "fluctuations of about 1 part in 100,000. These anisotropies encode information "
            "about the density, composition, and geometry of the early universe and were "
            "mapped in exquisite detail by the Planck satellite."
        ),
    },
    {
        "id": "star_formation_01",
        "text": (
            "Stars form within molecular clouds — vast regions of cold, dense gas and dust in "
            "the interstellar medium. When a region of a molecular cloud becomes gravitationally "
            "unstable (often triggered by a nearby supernova shockwave or cloud-cloud collision), "
            "it collapses under its own gravity, fragmenting into smaller clumps called protostars. "
            "As the protostar contracts, it heats up. When the core temperature reaches about "
            "10 million Kelvin, hydrogen fusion ignites and the object becomes a main-sequence "
            "star. The mass of the protostar determines its lifetime, luminosity, and eventual "
            "fate — low-mass stars become white dwarfs, while high-mass stars end as neutron "
            "stars or black holes."
        ),
    },
    {
        "id": "habitable_zone_01",
        "text": (
            "The habitable zone (or Goldilocks zone) is the region around a star where "
            "conditions are right for liquid water to exist on a planet's surface — neither "
            "too hot nor too cold. For the Sun, this zone extends from roughly 0.95 to 1.37 AU. "
            "However, habitability depends on many factors beyond distance: the planet's "
            "atmosphere, magnetic field, geological activity, and the star's variability all "
            "play critical roles. Tidally locked planets orbiting red dwarf stars, for example, "
            "may have habitable conditions on the terminator zone between their permanent day "
            "and night sides."
        ),
    },
    {
        "id": "gravitational_waves_01",
        "text": (
            "Gravitational waves are ripples in the fabric of spacetime, predicted by Einstein "
            "in 1916 and first directly detected by LIGO in September 2015 from the merger of "
            "two stellar-mass black holes (GW150914). These waves propagate at the speed of "
            "light and are generated by accelerating masses — particularly during violent cosmic "
            "events such as black hole mergers, neutron star collisions, and asymmetric "
            "supernovae. The LIGO and Virgo detectors use laser interferometry to measure "
            "spacetime distortions as small as 10^-21 meters. Gravitational wave astronomy has "
            "opened an entirely new observational window on the universe."
        ),
    },
    {
        "id": "james_webb_01",
        "text": (
            "The James Webb Space Telescope (JWST), launched in December 2021, is the most "
            "powerful space telescope ever built. Its 6.5-meter gold-coated primary mirror "
            "and suite of infrared instruments operate at the second Lagrange point (L2), "
            "1.5 million km from Earth. JWST is designed to observe the universe in the "
            "near- and mid-infrared spectrum, enabling it to see through dust clouds, study "
            "the atmospheres of exoplanets, and observe the most distant galaxies — some "
            "formed within 200 million years of the Big Bang. Early observations have already "
            "challenged existing models of galaxy formation and evolution."
        ),
    },
    {
        "id": "jupiter_moons_01",
        "text": (
            "Jupiter has at least 95 known moons, four of which — Io, Europa, Ganymede, and "
            "Callisto — are the Galilean moons discovered by Galileo in 1610. Europa is of "
            "particular interest to astrobiologists because it harbors a global subsurface "
            "ocean of liquid water beneath a thick ice shell. Tidal heating from Jupiter's "
            "immense gravity keeps this ocean liquid and may drive hydrothermal vents on the "
            "ocean floor — an environment analogous to Earth's deep-sea vents, which support "
            "life without sunlight. NASA's Europa Clipper mission, launched in 2024, aims to "
            "assess Europa's habitability through dozens of close flybys."
        ),
    },
    {
        "id": "supernova_01",
        "text": (
            "A supernova is the explosive death of a star, briefly outshining an entire galaxy. "
            "Type II supernovae occur when massive stars (>8 solar masses) exhaust their nuclear "
            "fuel; the iron core collapses in milliseconds, producing a neutron star or black "
            "hole and ejecting the outer layers at speeds up to 30,000 km/s. Type Ia supernovae "
            "occur when a white dwarf in a binary system accretes matter from its companion "
            "until it exceeds the Chandrasekhar limit (~1.4 solar masses) and undergoes "
            "thermonuclear detonation. Type Ia supernovae have remarkably consistent peak "
            "luminosities, making them invaluable as standard candles for measuring cosmic "
            "distances."
        ),
    },
    {
        "id": "solar_system_formation_01",
        "text": (
            "The solar system formed approximately 4.6 billion years ago from the gravitational "
            "collapse of a region within a large molecular cloud. The collapse produced a "
            "rotating disk of gas and dust called the solar nebula, with the proto-Sun at its "
            "center. In the inner solar system, temperatures were too high for volatiles to "
            "condense, so only rocky and metallic materials formed the terrestrial planets "
            "(Mercury, Venus, Earth, Mars). Beyond the frost line (~3 AU), water ice and other "
            "volatiles could condense, providing additional building material for the giant "
            "planets (Jupiter, Saturn, Uranus, Neptune)."
        ),
    },
    {
        "id": "redshift_01",
        "text": (
            "Cosmological redshift is the stretching of light waves caused by the expansion "
            "of the universe. As photons travel across expanding space, their wavelength "
            "increases — shifting visible light toward the red end of the spectrum. The "
            "redshift z of a distant object is directly related to how much the universe has "
            "expanded since the light was emitted: a galaxy at z=1 is seen as it was when "
            "the universe was half its current size. Edwin Hubble's 1929 observation that "
            "distant galaxies are receding faster than nearby ones (Hubble's Law) provided "
            "the first evidence for the expanding universe."
        ),
    },
    {
        "id": "tidal_locking_01",
        "text": (
            "Tidal locking occurs when a celestial body's orbital period matches its rotational "
            "period, so the same face always points toward its orbital partner. The Moon is "
            "tidally locked to Earth — we always see the same side. Tidal locking results from "
            "tidal friction: gravitational forces raise tidal bulges that gradually slow the "
            "body's rotation until it synchronizes with its orbit. Many exoplanets orbiting "
            "close to their host stars are expected to be tidally locked, creating extreme "
            "temperature differences between their permanent day and night hemispheres."
        ),
    },
    {
        "id": "spectroscopy_01",
        "text": (
            "Spectroscopy is the study of how matter interacts with electromagnetic radiation. "
            "In astronomy, spectroscopy reveals the chemical composition, temperature, density, "
            "and velocity of celestial objects. Each element produces a unique pattern of "
            "absorption or emission lines — a spectral fingerprint. By analyzing starlight, "
            "astronomers can determine what elements are present in a star's atmosphere, how "
            "fast the star is moving toward or away from us (via Doppler shift), and even "
            "detect the presence of exoplanet atmospheres by observing which wavelengths are "
            "absorbed during a transit."
        ),
    },
]

print(f"Knowledge base created: {len(knowledge_base)} documents")
for i, doc in enumerate(knowledge_base):
    print(f"  [{doc['id']}] {doc['text'][:70]}...")

## 7 · Building the Vector Store

We embed all documents with **BAAI/bge-base-en-v1.5** and store them in a **FAISS** index for fast similarity search. FAISS (Facebook AI Similarity Search) uses optimized algorithms for nearest-neighbor lookup in high-dimensional spaces.

In [ ]:
# Embed all documents and build FAISS index
texts = [doc["text"] for doc in knowledge_base]
doc_embeddings = embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

# Build FAISS index (Inner Product = cosine similarity when vectors are normalized)
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings.astype(np.float32))

print(f"✓ FAISS index built")
print(f"  Documents indexed: {index.ntotal}")
print(f"  Embedding dimension: {dimension}")
print(f"  Index type: Flat Inner Product (exact cosine search)")

## 8 · Standard Retrieval (Baseline)

First, let's implement standard dense retrieval: embed the query directly and find the nearest documents. This is what we want to beat with HyDE.

In [ ]:
def standard_retrieve(query, k=3):
    """Standard dense retrieval: embed the query directly."""
    query_embedding = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(query_embedding.astype(np.float32), k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "id": knowledge_base[idx]["id"],
            "text": knowledge_base[idx]["text"],
            "score": float(score),
        })
    return results

# Test it
query = "What causes the Northern Lights?"
results = standard_retrieve(query, k=3)

print(f"Query: '{query}'")
print(f"\n{'='*60}")
print("STANDARD RETRIEVAL — Top 3 Results")
print(f"{'='*60}")
for i, r in enumerate(results):
    print(f"\n#{i+1} [{r['id']}] (score: {r['score']:.4f})")
    print(f"   {r['text'][:120]}...")

## 9 · Implementing HyDE from Scratch

Now the core of this notebook: we implement HyDE step by step.

### Step 1: Generate a Hypothetical Document

We prompt the LLM to write a passage that **would** answer the query. The prompt explicitly tells the model to write in a document-like style, matching the format of our knowledge base.

In [ ]:
def generate_hypothetical_document(query, max_tokens=512):
    """Generate a hypothetical document that would answer the query."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a scientific reference writer. Given a question, write a detailed "
                "factual passage (150-250 words) that would appear in a reference document "
                "answering this question. Write in an expository, encyclopedic style. "
                "Include specific details, numbers, and technical terms. "
                "Do NOT format as Q&A — write as a standalone informational passage."
            ),
        },
        {
            "role": "user",
            "content": f"Write a reference passage that answers: {query}",
        },
    ]
    result = generate(messages, max_new_tokens=max_tokens, temperature=0.7, thinking=False)
    return result

# Test hypothetical document generation
query = "What causes the Northern Lights?"
hypo_doc = generate_hypothetical_document(query)

print("QUERY:", query)
print(f"\n{'='*60}")
print("GENERATED HYPOTHETICAL DOCUMENT:")
print(f"{'='*60}")
print(hypo_doc)
print(f"\n[Length: {len(hypo_doc.split())} words]")

### Step 2: Embed the Hypothetical Document and Retrieve

Now we embed the hypothetical document instead of the query and search for the nearest real documents. This is the core HyDE operation.

In [ ]:
def hyde_retrieve(query, k=3, show_hypothetical=True):
    """HyDE retrieval: generate hypothetical doc, embed it, search."""
    # Step 1: Generate hypothetical document
    hypo_doc = generate_hypothetical_document(query)

    # Step 2: Embed the hypothetical document
    hypo_embedding = embed_model.encode([hypo_doc], normalize_embeddings=True)

    # Step 3: Search with the hypothetical embedding
    scores, indices = index.search(hypo_embedding.astype(np.float32), k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "id": knowledge_base[idx]["id"],
            "text": knowledge_base[idx]["text"],
            "score": float(score),
        })

    if show_hypothetical:
        print("HYPOTHETICAL DOCUMENT (used for search):")
        print("-" * 40)
        print(hypo_doc[:300] + ("..." if len(hypo_doc) > 300 else ""))
        print("-" * 40)
        print()

    return results, hypo_doc

# Test HyDE retrieval
query = "What causes the Northern Lights?"
results, hypo = hyde_retrieve(query, k=3)

print(f"Query: '{query}'")
print(f"\n{'='*60}")
print("HyDE RETRIEVAL — Top 3 Results")
print(f"{'='*60}")
for i, r in enumerate(results):
    print(f"\n#{i+1} [{r['id']}] (score: {r['score']:.4f})")
    print(f"   {r['text'][:120]}...")

### The Hypothetical Document May Contain Errors — And That's OK!

Let's look at a case where the hypothetical document has factual mistakes but *still* leads to correct retrieval. This demonstrates why HyDE is robust: the embedding captures **topical similarity**, not factual correctness.

In [ ]:
# Ask about something where the LLM might hallucinate details
query = "How do we detect gravitational waves?"
hypo_doc = generate_hypothetical_document(query)

print("HYPOTHETICAL DOCUMENT:")
print("="*60)
print(hypo_doc)
print("="*60)

# Now embed it and retrieve
hypo_emb = embed_model.encode([hypo_doc], normalize_embeddings=True)
scores, indices = index.search(hypo_emb.astype(np.float32), 3)

print(f"\nRetrieval results (does it find gravitational_waves_01?):")
for score, idx in zip(scores[0], indices[0]):
    doc = knowledge_base[idx]
    print(f"  [{doc['id']}] score={score:.4f} — {doc['text'][:80]}...")

print("\n✓ Even if the hypothetical doc has imprecise details,")
print("  it still retrieves the correct real document!")

## 10 · Embedding Space Visualization

Let's visualize **why** HyDE works by computing pairwise cosine similarities between the query, hypothetical document, and all real documents. This makes the geometric argument concrete.

In [ ]:
# Compute detailed similarity analysis
query = "What causes the Northern Lights?"
hypo_doc = generate_hypothetical_document(query)

# Encode everything
query_emb = embed_model.encode([query], normalize_embeddings=True)
hypo_emb = embed_model.encode([hypo_doc], normalize_embeddings=True)

# Cosine similarities (with normalized vectors, dot product = cosine similarity)
query_vs_docs = (query_emb @ doc_embeddings.T).flatten()
hypo_vs_docs = (hypo_emb @ doc_embeddings.T).flatten()

# Sort by HyDE score to show ranking difference
hyde_ranking = np.argsort(-hypo_vs_docs)
standard_ranking = np.argsort(-query_vs_docs)

print(f"Query: '{query}'")
print(f"\n{'='*70}")
print(f"{'Document':<30} {'Std Score':>10} {'HyDE Score':>10} {'Δ':>8}")
print(f"{'='*70}")
for idx in hyde_ranking[:8]:
    doc_id = knowledge_base[idx]["id"]
    std_s = query_vs_docs[idx]
    hyde_s = hypo_vs_docs[idx]
    delta = hyde_s - std_s
    marker = " ← target" if "aurora" in doc_id else ""
    print(f"  {doc_id:<28} {std_s:>10.4f} {hyde_s:>10.4f} {delta:>+8.4f}{marker}")

print(f"\n{'='*70}")
print("KEY OBSERVATIONS:")
print("  • HyDE scores are generally higher for relevant documents")
print("  • The hypothetical doc (a document) is closer to other documents")
print("  • Query-doc similarity is limited by the query-document gap")

## 11 · Head-to-Head: Standard vs HyDE Retrieval

Let's run both methods on the same queries and compare their top results. This is the most direct way to see HyDE's advantage.

In [ ]:
def compare_retrieval(query, k=3):
    """Compare standard vs HyDE retrieval on the same query."""
    # Standard retrieval
    std_results = standard_retrieve(query, k=k)

    # HyDE retrieval
    hyde_results, hypo_doc = hyde_retrieve(query, k=k, show_hypothetical=False)

    return std_results, hyde_results, hypo_doc

test_queries = [
    "What causes the Northern Lights?",
    "How do scientists find planets around other stars?",
    "Why is the universe expanding faster and faster?",
    "What happens when a massive star dies?",
    "How did the solar system form?",
]

for query in test_queries:
    std, hyde, hypo = compare_retrieval(query, k=3)
    print(f"\nQuery: '{query}'")
    print(f"  Standard top-1: [{std[0]['id']}] score={std[0]['score']:.4f}")
    print(f"  HyDE    top-1: [{hyde[0]['id']}] score={hyde[0]['score']:.4f}")

    # Check if top-3 sets differ
    std_ids = {r["id"] for r in std}
    hyde_ids = {r["id"] for r in hyde}
    if std_ids != hyde_ids:
        only_hyde = hyde_ids - std_ids
        only_std = std_ids - hyde_ids
        if only_hyde:
            print(f"  → HyDE found: {only_hyde} (not in standard results)")
        if only_std:
            print(f"  → Standard found: {only_std} (not in HyDE results)")
    else:
        print(f"  → Same documents retrieved (but scores differ)")
    print()

## 12 · Multiple Hypothetical Documents — Ensemble HyDE

A single hypothetical document captures *one possible way* to answer the query. But the LLM might emphasize different aspects each time. By generating **N hypothetical documents** and averaging their embeddings, we get a more robust search vector that covers multiple angles.

### Why Averaging Works

Each hypothetical document is a "vote" for a region of embedding space. Averaging N embeddings produces a centroid that is:
- Closer to the **consensus region** across different phrasings
- Less sensitive to any single hallucination
- More robust to LLM sampling randomness

In [ ]:
def hyde_retrieve_ensemble(query, n_hypotheticals=3, k=3):
    """Generate N hypothetical documents, average their embeddings, search."""
    print(f"Generating {n_hypotheticals} hypothetical documents...")

    hypo_docs = []
    hypo_embeddings = []

    for i in range(n_hypotheticals):
        doc = generate_hypothetical_document(query)
        emb = embed_model.encode([doc], normalize_embeddings=True)
        hypo_docs.append(doc)
        hypo_embeddings.append(emb[0])
        print(f"  Doc {i+1}: {doc[:80]}...")

    # Average and re-normalize
    avg_embedding = np.mean(hypo_embeddings, axis=0, keepdims=True)
    avg_embedding = avg_embedding / np.linalg.norm(avg_embedding, axis=1, keepdims=True)

    # Search
    scores, indices = index.search(avg_embedding.astype(np.float32), k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "id": knowledge_base[idx]["id"],
            "text": knowledge_base[idx]["text"],
            "score": float(score),
        })

    return results, hypo_docs

# Test ensemble HyDE
query = "What is the leftover radiation from the early universe?"
ensemble_results, hypo_docs = hyde_retrieve_ensemble(query, n_hypotheticals=3, k=3)

print(f"\n{'='*60}")
print(f"ENSEMBLE HyDE — Top 3 Results for: '{query}'")
print(f"{'='*60}")
for i, r in enumerate(ensemble_results):
    print(f"#{i+1} [{r['id']}] (score: {r['score']:.4f})")
    print(f"   {r['text'][:120]}...")
    print()

In [ ]:
# Compare single HyDE vs ensemble HyDE
query = "What is the leftover radiation from the early universe?"

# Single HyDE
single_results, _ = hyde_retrieve(query, k=3, show_hypothetical=False)

# Ensemble HyDE (already computed above)
# Let's also compute standard for comparison
std_results = standard_retrieve(query, k=3)

print(f"Query: '{query}'")
print(f"\n{'Method':<20} {'Top-1 Document':<30} {'Score':>8}")
print("="*60)
print(f"  {'Standard':<18} {std_results[0]['id']:<30} {std_results[0]['score']:>8.4f}")
print(f"  {'HyDE (single)':<18} {single_results[0]['id']:<30} {single_results[0]['score']:>8.4f}")
print(f"  {'HyDE (ensemble)':<18} {ensemble_results[0]['id']:<30} {ensemble_results[0]['score']:>8.4f}")
print()
print("Ensemble HyDE often achieves the highest similarity score because")
print("averaging multiple hypotheticals reduces noise and captures broader context.")

## 13 · Complete RAG Pipeline with HyDE

Now let's build a full **Retrieval-Augmented Generation** pipeline that uses HyDE for the retrieval step. The pipeline:

1. Takes a user question
2. Generates a hypothetical document (HyDE)
3. Retrieves real documents using the hypothetical embedding
4. Generates a final answer grounded in the retrieved documents

In [ ]:
def hyde_rag(query, k=3, n_hypotheticals=1, verbose=True):
    """Complete RAG pipeline with HyDE retrieval."""
    if verbose:
        print(f"Question: {query}")
        print(f"\n{'─'*60}")

    # Step 1: HyDE Retrieval
    if n_hypotheticals > 1:
        retrieved, hypo_docs = hyde_retrieve_ensemble(query, n_hypotheticals=n_hypotheticals, k=k)
    else:
        retrieved, hypo_doc = hyde_retrieve(query, k=k, show_hypothetical=False)
        hypo_docs = [hypo_doc]

    if verbose:
        print(f"\nRetrieved {len(retrieved)} documents:")
        for r in retrieved:
            print(f"  [{r['id']}] score={r['score']:.4f}")
        print(f"{'─'*60}")

    # Step 2: Generate answer from REAL documents
    context = "\n\n---\n\n".join([r["text"] for r in retrieved])
    messages = [
        {
            "role": "system",
            "content": (
                "You are a knowledgeable science assistant. Answer the user's question "
                "based ONLY on the provided context. If the context doesn't contain "
                "enough information, say so. Be specific and cite details from the context."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}",
        },
    ]

    _, answer = generate(messages, max_new_tokens=512, temperature=0.3, thinking=True)

    if verbose:
        print(f"\nFINAL ANSWER (grounded in real documents):")
        print(f"{'─'*60}")
        print(answer)

    return answer, retrieved, hypo_docs

# Test the full pipeline
answer, docs, hypos = hyde_rag("What causes the Northern Lights?")

### Testing the Complete Pipeline on Multiple Queries

In [ ]:
questions = [
    "How do scientists detect planets orbiting distant stars?",
    "What are gravitational waves and how were they discovered?",
    "Why does Mars have such a thin atmosphere?",
]

for q in questions:
    print(f"\n{'='*60}")
    answer, docs, _ = hyde_rag(q, verbose=True)
    print(f"{'='*60}")

## 14 · When HyDE Fails

HyDE is not a silver bullet. It can fail in specific scenarios:

### 1. Very Domain-Specific Queries
When the query requires **highly specialized knowledge** that the LLM doesn't have, the hypothetical document will be too generic or outright wrong — and its embedding may point to the wrong region.

### 2. Ambiguous Queries
If the query could mean multiple things, the hypothetical document commits to one interpretation, potentially missing relevant documents about other interpretations.

### 3. Factoid Queries Seeking Exact Entities
For queries like *"What is the name of Jupiter's largest moon?"*, the LLM might hallucinate a wrong name. While this often doesn't matter (the embedding still captures "Jupiter moons"), it can occasionally mislead retrieval.

### 4. Added Latency
HyDE requires an LLM call before retrieval, adding latency. For real-time applications, this extra step may not be acceptable.

### 5. Knowledge Base Too Small or Too Uniform
If all documents are very similar (e.g., all about the same narrow topic), HyDE's advantage diminishes because standard retrieval already works well.

In [ ]:
# Demonstrate a failure case: query about something NOT in the knowledge base
query_outside_kb = "What is the chemical composition of Titan's hydrocarbon lakes?"

print("FAILURE CASE 1: Query outside knowledge base scope")
print(f"Query: '{query_outside_kb}'")
print()

# The LLM will generate a hypothetical about Titan's lakes
hypo_doc = generate_hypothetical_document(query_outside_kb)
print(f"Hypothetical doc (first 200 chars): {hypo_doc[:200]}...")

# But our KB has nothing about Titan's lakes — retrieval returns irrelevant docs
results, _ = hyde_retrieve(query_outside_kb, k=3, show_hypothetical=False)
print(f"\nRetrieved documents (none are actually about Titan's lakes):")
for r in results:
    print(f"  [{r['id']}] score={r['score']:.4f} — {r['text'][:80]}...")

print()
print("FAILURE CASE 2: Ambiguous query")
query_ambiguous = "Tell me about the light"
print(f"Query: '{query_ambiguous}'")

# Standard retrieval
std = standard_retrieve(query_ambiguous, k=3)
hyde_res, hypo = hyde_retrieve(query_ambiguous, k=3, show_hypothetical=False)

print(f"\nStandard retrieval — diverse results:")
for r in std:
    print(f"  [{r['id']}] score={r['score']:.4f}")

print(f"\nHyDE retrieval — may commit to one interpretation:")
for r in hyde_res:
    print(f"  [{r['id']}] score={r['score']:.4f}")

print(f"\nHyDE hypothetical committed to: {hypo[:100]}...")
print("This shows HyDE's disambiguation can be a double-edged sword.")

## 15 · Quantitative Comparison — Standard vs HyDE

Let's run a systematic comparison across all our test queries. For each query, we measure the cosine similarity of the top retrieved document. Higher similarity generally indicates better retrieval quality.

In [ ]:
comparison_queries = [
    ("What causes the Northern Lights?", "aurora_01"),
    ("How do scientists find planets around other stars?", "exoplanet_detection_01"),
    ("What happens inside a collapsing massive star?", "supernova_01"),
    ("What is the evidence for invisible matter in galaxies?", "dark_matter_01"),
    ("What did the CMB reveal about the early universe?", "cosmic_microwave_background_01"),
    ("How does tidal friction affect planetary rotation?", "tidal_locking_01"),
    ("What telescope orbits at the L2 Lagrange point?", "james_webb_01"),
    ("How fast does the solar wind travel?", "solar_wind_01"),
]

print(f"{'Query':<55} {'Expected':<20} {'Std Top-1':<20} {'HyDE Top-1':<20} {'Winner'}")
print("="*120)

std_wins = 0
hyde_wins = 0
ties = 0

for query, expected_id in comparison_queries:
    std = standard_retrieve(query, k=1)
    hyde_res, _ = hyde_retrieve(query, k=1, show_hypothetical=False)

    std_id = std[0]["id"]
    hyde_id = hyde_res[0]["id"]

    std_correct = "✓" if std_id == expected_id else "✗"
    hyde_correct = "✓" if hyde_id == expected_id else "✗"

    if hyde_id == expected_id and std_id != expected_id:
        winner = "HyDE ✓"
        hyde_wins += 1
    elif std_id == expected_id and hyde_id != expected_id:
        winner = "Std ✓"
        std_wins += 1
    elif std_id == expected_id and hyde_id == expected_id:
        winner = "Tie"
        ties += 1
    else:
        winner = "Neither"

    q_short = query[:52] + "..." if len(query) > 55 else query
    print(f"  {q_short:<53} {expected_id:<20} {std_id} {std_correct:<5}  {hyde_id} {hyde_correct:<5} {winner}")

print(f"\n{'='*120}")
print(f"RESULTS: Standard wins={std_wins}, HyDE wins={hyde_wins}, Ties={ties}")
print(f"HyDE win rate: {(hyde_wins + ties) / len(comparison_queries) * 100:.0f}% (wins + ties)")

## 16 · Similarity Score Distribution

Let's visualize the distribution of similarity scores to see how HyDE shifts the score distribution upward. Higher scores mean better matches.

In [ ]:
# Compute average similarity scores across all queries
all_std_top1_scores = []
all_hyde_top1_scores = []
all_std_top3_avg = []
all_hyde_top3_avg = []

analysis_queries = [
    "What causes the Northern Lights?",
    "How do scientists find planets around other stars?",
    "What happens inside a collapsing massive star?",
    "What is the evidence for invisible matter in galaxies?",
    "What did the CMB reveal about the early universe?",
    "How does tidal friction affect planetary rotation?",
]

for query in analysis_queries:
    std = standard_retrieve(query, k=3)
    hyde_res, _ = hyde_retrieve(query, k=3, show_hypothetical=False)

    all_std_top1_scores.append(std[0]["score"])
    all_hyde_top1_scores.append(hyde_res[0]["score"])
    all_std_top3_avg.append(np.mean([r["score"] for r in std]))
    all_hyde_top3_avg.append(np.mean([r["score"] for r in hyde_res]))

print("SIMILARITY SCORE ANALYSIS")
print("="*50)
print(f"\nTop-1 Scores:")
print(f"  Standard — Mean: {np.mean(all_std_top1_scores):.4f}, Min: {np.min(all_std_top1_scores):.4f}, Max: {np.max(all_std_top1_scores):.4f}")
print(f"  HyDE     — Mean: {np.mean(all_hyde_top1_scores):.4f}, Min: {np.min(all_hyde_top1_scores):.4f}, Max: {np.max(all_hyde_top1_scores):.4f}")
print(f"  Δ (HyDE - Std):  {np.mean(all_hyde_top1_scores) - np.mean(all_std_top1_scores):+.4f}")

print(f"\nTop-3 Average Scores:")
print(f"  Standard — Mean: {np.mean(all_std_top3_avg):.4f}")
print(f"  HyDE     — Mean: {np.mean(all_hyde_top3_avg):.4f}")
print(f"  Δ (HyDE - Std):  {np.mean(all_hyde_top3_avg) - np.mean(all_std_top3_avg):+.4f}")

print("\n" + "="*50)
print("Per-query breakdown:")
for i, q in enumerate(analysis_queries):
    delta = all_hyde_top1_scores[i] - all_std_top1_scores[i]
    bar = "▓" * int(abs(delta) * 200)
    direction = "+" if delta > 0 else "-"
    print(f"  {q[:45]:<47} Δ={delta:+.4f} {bar}")

## 17 · Latency Trade-off

HyDE adds an LLM generation step before retrieval. Let's measure the actual cost.

In [ ]:
import time

query = "How do neutron stars form?"

# Standard retrieval timing
start = time.time()
std_results = standard_retrieve(query, k=3)
std_time = time.time() - start

# HyDE retrieval timing (includes LLM generation)
start = time.time()
hyde_results, hypo_doc = hyde_retrieve(query, k=3, show_hypothetical=False)
hyde_time = time.time() - start

# HyDE ensemble timing
start = time.time()
ensemble_results, hypo_docs = hyde_retrieve_ensemble(query, n_hypotheticals=3, k=3)
ensemble_time = time.time() - start

print("LATENCY COMPARISON")
print("="*50)
print(f"  Standard retrieval:      {std_time*1000:>8.1f} ms")
print(f"  HyDE (1 hypothetical):   {hyde_time*1000:>8.1f} ms")
print(f"  HyDE (3 hypotheticals):  {ensemble_time*1000:>8.1f} ms")
print(f"\n  Overhead (single HyDE):  {(hyde_time - std_time)*1000:>8.1f} ms")
print(f"  Overhead (ensemble):     {(ensemble_time - std_time)*1000:>8.1f} ms")
print(f"\nThe overhead is entirely from LLM generation.")
print(f"Embedding + FAISS search adds < 10ms regardless of method.")

## 18 · When to Use HyDE — Decision Framework

| Scenario | Standard Retrieval | HyDE | Why? |
|---|---|---|---|
| Short, vague queries | ⚠️ May miss relevant docs | ✅ LLM expands to full doc | Bridges query-document gap |
| Out-of-domain queries | ⚠️ Poor embedding match | ✅ Better topical embedding | LLM has broad knowledge |
| Very specific factoid queries | ✅ Fast and sufficient | ⚠️ Overkill, adds latency | Short queries can match specific terms well |
| Real-time applications | ✅ Low latency | ⚠️ LLM adds 1-5s | Latency budget matters |
| Small knowledge base | ✅ Works fine | ⚠️ Less benefit | Gap is smaller when docs are few |
| Multi-faceted complex queries | ⚠️ Captures one aspect | ✅ (ensemble) Covers angles | Multiple hypotheticals = multiple perspectives |

### Rules of Thumb

1. **Use HyDE** when queries are natural-language questions and your knowledge base has long, detailed documents
2. **Use Ensemble HyDE** for complex queries where multiple aspects need to be captured
3. **Skip HyDE** for keyword-like queries, very small knowledge bases, or when latency is critical
4. **Always** generate the final answer from *real* documents, never from the hypothetical one

## 19 · Advanced: Prompt Engineering for Hypothetical Documents

The quality of the hypothetical document depends on the prompt. Let's explore different prompt strategies and see how they affect retrieval.

In [ ]:
def generate_hypo_with_style(query, style="encyclopedic"):
    """Generate hypothetical documents with different prompt styles."""
    style_prompts = {
        "encyclopedic": (
            "Write a detailed encyclopedia entry (150-200 words) that answers "
            "this question. Use formal, technical language with specific facts."
        ),
        "textbook": (
            "Write a textbook paragraph (150-200 words) explaining the answer "
            "to this question. Include definitions and use pedagogical tone."
        ),
        "brief": (
            "Write a brief factual summary (50-80 words) answering this question. "
            "Be concise but include key technical terms."
        ),
    }

    messages = [
        {"role": "system", "content": style_prompts[style]},
        {"role": "user", "content": f"Question: {query}"},
    ]
    return generate(messages, max_new_tokens=400, temperature=0.7, thinking=False)

# Compare styles
query = "What are neutron stars?"

print(f"Query: '{query}'\n")
for style in ["encyclopedic", "textbook", "brief"]:
    hypo = generate_hypo_with_style(query, style)
    hypo_emb = embed_model.encode([hypo], normalize_embeddings=True)
    scores, indices = index.search(hypo_emb.astype(np.float32), 1)

    print(f"Style: {style}")
    print(f"  Hypothetical ({len(hypo.split())} words): {hypo[:100]}...")
    print(f"  Top-1: [{knowledge_base[indices[0][0]]['id']}] score={scores[0][0]:.4f}")
    print()

## 20 · Putting It All Together — The HyDE RAG Class

Here's a clean, reusable implementation wrapping everything we've built.

In [ ]:
class HyDERAG:
    """Complete HyDE-based RAG system."""

    def __init__(self, documents, embed_model, faiss_index):
        self.documents = documents
        self.embed_model = embed_model
        self.index = faiss_index

    def _generate_hypothetical(self, query, n=1):
        """Generate n hypothetical documents for a query."""
        hypos = []
        for _ in range(n):
            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are a reference writer. Given a question, write a detailed "
                        "factual passage (150-250 words) that would appear in an "
                        "authoritative document answering this question. "
                        "Write in expository style. Do NOT format as Q&A."
                    ),
                },
                {"role": "user", "content": f"Write a passage answering: {query}"},
            ]
            hypos.append(generate(messages, max_new_tokens=512, temperature=0.7, thinking=False))
        return hypos

    def retrieve(self, query, k=3, n_hypotheticals=1):
        """Retrieve documents using HyDE."""
        hypos = self._generate_hypothetical(query, n=n_hypotheticals)
        embeddings = self.embed_model.encode(hypos, normalize_embeddings=True)

        # Average if multiple
        if len(embeddings) > 1:
            avg = np.mean(embeddings, axis=0, keepdims=True)
            avg = avg / np.linalg.norm(avg, axis=1, keepdims=True)
        else:
            avg = embeddings

        scores, indices = self.index.search(avg.astype(np.float32), k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            results.append({
                "id": self.documents[idx]["id"],
                "text": self.documents[idx]["text"],
                "score": float(score),
            })
        return results, hypos

    def query(self, question, k=3, n_hypotheticals=1):
        """Full RAG: retrieve then generate answer."""
        retrieved, hypos = self.retrieve(question, k=k, n_hypotheticals=n_hypotheticals)
        context = "\n\n".join([r["text"] for r in retrieved])

        messages = [
            {
                "role": "system",
                "content": (
                    "Answer the question based ONLY on the context provided. "
                    "Be specific and reference details from the context."
                ),
            },
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ]
        _, answer = generate(messages, max_new_tokens=512, temperature=0.3, thinking=True)
        return {"answer": answer, "retrieved": retrieved, "hypotheticals": hypos}

# Instantiate and test
rag = HyDERAG(knowledge_base, embed_model, index)
result = rag.query("How did LIGO detect gravitational waves?", k=3)

print("Question: How did LIGO detect gravitational waves?")
print(f"\nRetrieved: {[r['id'] for r in result['retrieved']]}")
print(f"\nAnswer:\n{result['answer']}")

## Key Takeaways

### What We Learned

1. **The Query-Document Gap is Real** — Short queries and long documents live in different regions of embedding space. Cosine similarity between a 7-word question and a 300-word answer is inherently limited.

2. **HyDE Bridges the Gap** — By generating a hypothetical document, we transform the search query from "query space" into "document space." The hypothetical document uses similar vocabulary, length, and style as real documents.

3. **Factual Accuracy Doesn't Matter (for Retrieval)** — The hypothetical document may contain hallucinations. That's fine — its job is to land near the right documents in embedding space, not to be correct.

4. **Ensemble HyDE is More Robust** — Generating multiple hypotheticals and averaging their embeddings reduces noise and covers more aspects of the query.

5. **HyDE Has a Latency Cost** — The LLM generation step adds 1-5 seconds. Use HyDE when retrieval quality matters more than speed.

6. **HyDE Fails Gracefully** — When the LLM's hypothetical is off-topic, retrieval falls back to roughly standard quality (the hypothetical is no worse than a random document embedding).

### The Big Idea

> HyDE turns the LLM into a **query expansion engine** that speaks the same language as the document index. It's a beautifully simple idea: **don't search with questions — search with answers.**